# GPU 双语字幕运行器（可迁移版）

支持具备 NVIDIA CUDA GPU 的 Kaggle、ModelScope DSW 和其他 Linux 云算力。Whisper 使用 CUDA FP16，Hy-MT2 通过 llama.cpp 卸载至 GPU。

In [ ]:
from pathlib import Path
import os
import subprocess

# 不写死平台路径：默认使用当前工作目录下的 asr。
# 在不同云平台可先设置 os.environ['ASR_WORKDIR'] = '/持久化目录/asr'。
WORKDIR = Path(os.environ.get('ASR_WORKDIR', Path.cwd() / 'asr')).expanduser()
WORKDIR.mkdir(parents=True, exist_ok=True)
%cd {WORKDIR}
print(f'工作目录：{WORKDIR}')

def run(command):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, check=True)

# 每次运行均从 GitHub 的 main 分支获取最新核心处理脚本。
SCRIPT = WORKDIR / 'transcribe_translate_bilingual_gpu.py'
SCRIPT_URL = 'https://raw.githubusercontent.com/brad1127/asr-transcribe-translate/main/transcribe_translate_bilingual_gpu.py'
run(['wget', '-q', '-O', str(SCRIPT), SCRIPT_URL])
assert SCRIPT.is_file() and SCRIPT.stat().st_size > 0, 'GitHub 脚本下载失败'
print(f'已同步最新脚本：{SCRIPT}')

run(['python', '-m', 'pip', 'install', '-U', 'pip'])
run(['python', '-m', 'pip', 'install', 'faster-whisper', 'av', 'ctranslate2', 'tokenizers', 'onnxruntime', 'numpy', 'requests', 'tqdm', 'huggingface_hub'])
run(['nvidia-smi'])

llama_dir = WORKDIR / 'llama_cpp'
archive = WORKDIR / 'llama.cpp.tar.gz'
llama_dir.mkdir(exist_ok=True)
if not any(path.is_file() for path in llama_dir.rglob('llama-server')):
    run(['wget', '-c', '--show-progress', '--timeout=30', '--tries=3', 'https://github.com/ggml-org/llama.cpp/releases/download/b10516/llama-b10516-bin-ubuntu-x64.tar.gz', '-O', str(archive)])
    run(['tar', '-xzf', str(archive), '-C', str(llama_dir)])
    archive.unlink(missing_ok=True)

servers = [path for path in llama_dir.rglob('llama-server') if path.is_file()]
assert servers, '未找到 llama-server'
servers[0].chmod(0o755)
print(f'准备完成：{servers[0]}')


## 自动下载公开模型
在 Kaggle 中请先在 Session options 开启 Internet；已存在的模型会自动跳过下载。

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

WHISPER_MODEL = WORKDIR / 'whisper' / 'medium'
TRANSLATION_MODEL = WORKDIR / 'models' / 'Hy-MT2-1.8B-GGUF' / 'Hy-MT2-1.8B-Q4_K_M.gguf'

if not (WHISPER_MODEL / 'model.bin').exists():
    snapshot_download(repo_id='Systran/faster-whisper-medium', local_dir=str(WHISPER_MODEL))
else:
    print(f'Whisper 模型已存在：{WHISPER_MODEL}')

if not TRANSLATION_MODEL.exists():
    TRANSLATION_MODEL.parent.mkdir(parents=True, exist_ok=True)
    hf_hub_download(repo_id='tencent/Hy-MT2-1.8B-GGUF', filename='Hy-MT2-1.8B-Q4_K_M.gguf', local_dir=str(TRANSLATION_MODEL.parent))
else:
    print(f'翻译模型已存在：{TRANSLATION_MODEL}')


## 检查脚本和视频
GPU 核心脚本会从 GitHub 同步；将 MP4 文件放进上述工作目录的 `videos/`。

In [ ]:
SCRIPT = WORKDIR / 'transcribe_translate_bilingual_gpu.py'
LLAMA_SERVER = next(path for path in (WORKDIR / 'llama_cpp').rglob('llama-server') if path.is_file())
VIDEO_DIR = Path("/kaggle/input/datasets/wangzhi2003cn163com/videos")
OUTPUT_DIR = WORKDIR / 'outputs'

for item in (WHISPER_MODEL, TRANSLATION_MODEL, SCRIPT, LLAMA_SERVER, VIDEO_DIR):
    print(('OK   ' if item.exists() else 'MISS '), item)
assert all(item.exists() for item in (WHISPER_MODEL, TRANSLATION_MODEL, SCRIPT, LLAMA_SERVER, VIDEO_DIR)), '请先放入脚本和视频。'


In [ ]:
videos = sorted(VIDEO_DIR.glob('*.mp4'))
assert videos, f'未在 {VIDEO_DIR} 找到 MP4 文件'

failed = []
for index, video in enumerate(videos, 1):
    output_dir = OUTPUT_DIR / video.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    command = ['python', str(SCRIPT), '--input', str(video), '--output-dir', str(output_dir), '--whisper-model', str(WHISPER_MODEL), '--translation-model', str(TRANSLATION_MODEL), '--llama-server', str(LLAMA_SERVER), '--threads', '4', '--whisper-device', 'cuda', '--whisper-compute-type', 'float16', '--gpu-layers', '999']
    print(f'[{index}/{len(videos)}] 开始：{video.name}')
    try:
        subprocess.run(command, check=True)
        print(f'完成：{video.name}')
    except subprocess.CalledProcessError:
        failed.append(video.name)
        print(f'失败：{video.name}')

if failed:
    raise RuntimeError(f'失败的视频：{failed}')
print('全部视频处理完成。')


In [ ]:
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path.name, f'{path.stat().st_size / 1024:.1f} KB')
